# Topic B — L2: Trees & Time-Dependent Volatility

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

**Prerequisites:** [Topic 0 — Foundations](Topic-0_Foundations.ipynb) (§3 risk-neutral pricing, §7 Black-Scholes).

**Exam map:** Practice-Final P2 (integrated variance), HW2 P1 (barrier tree), HW2 P2 (variance interpolation), Practice-Midterm P2 (discrete barrier tree).


In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)

def bs_call(S, K, T, r, sigma):
    if T <= 0: return max(S-K,0)
    std = sigma*np.sqrt(T)
    d1 = (np.log(S/K)+(r+0.5*sigma**2)*T)/std
    d2 = d1-std
    return S*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)

def trapz(y,x):
    return np.trapezoid(y,x) if hasattr(np,'trapezoid') else np.trapz(y,x)

## 1. Why trees?

Topic 0 gave $\text{Price}=e^{-rT}E^{\mathbb Q}[\text{payoff}]$. Trees approximate that expectation when $S_T$ has infinitely many possible values under GBM.

**Plain English:** Discretize time and stock price. Move backwards from expiry, taking discounted probability-weighted averages — risk-neutral pricing on a grid.


## 2. Backward induction (European)

At expiry: $C_N^j = f(S_N^j)$.

One step back:
$$C_n^j = e^{-r\Delta t}\bigl[p\,C_{n+1}^{j+1}+(1-p)\,C_{n+1}^{j-1}\bigr].$$

$p$ is chosen so the **stock** satisfies the same recursion (risk-neutral martingale).


In [2]:
S0,u,d,K,r,dt=100,1.1,0.9,100,0,0.5
ST=np.array([121,99,81]); CT=np.maximum(ST-K,0)
C_up=0.5*CT[0]+0.5*CT[1]; C_down=0.5*CT[1]+0.5*CT[2]
C0=0.5*C_up+0.5*C_down
print(f'2-step binomial call price: {C0:.4f}')

2-step binomial call price: 5.2500


## 3. Americans and barriers

**American put:** $C_n^j = \max\bigl((K-S_n^j)^+,\; e^{-r\Delta t}E[C_{n+1}]\bigr)$.

**Up-and-out put** (barrier $H$, monitored each step): if $S_n^j \ge H$ at a monitoring date, set $C_n^j=0$; else use European formula.

### Practice-Midterm P2(a) — full tree

$S_0=10$, each step $\pm 2$, $K=9$, $H=11$, $r=0$, 3 steps.

| Time | Stock levels | Option value (after barrier) |
|---|---|---|
| t=3 | 16,12,8,4 | 0,0,1,5 |
| t=2 | 14,10,6 | 0, 0.5, 3 |
| t=1 | 12,8 | 0, 1.75 |
| t=0 | 10 | **0.875** |


In [3]:
def upandout(S0,K,H,steps,r=0):
    levels={0:[S0]}
    for n in range(1,steps+1):
        levels[n]=sorted(set([s+2 for s in levels[n-1]]+[s-2 for s in levels[n-1]]))
    C={}
    for s in levels[steps]:
        C[(steps,s)]=max(K-s,0) if s<H else 0
    for n in range(steps-1,-1,-1):
        for s in levels[n]:
            if s>=H: C[(n,s)]=0
            else:
                vu=C.get((n+1,s+2),0); vd=C.get((n+1,s-2),0)
                C[(n,s)]=np.exp(-r)*0.5*(vu+vd)
    return C[(0,S0)]
print(f'Up-and-out put: {upandout(10,9,11,3):.4f}')

Up-and-out put: 0.8750


## 4. Trinomial trees (L2.21)

Log-price $X=\log S$ moves by $\pm\Delta x$ or 0. Match mean $\nu\Delta t$ and variance $\sigma^2\Delta t$:

$$p_u=\tfrac12\Bigl[\tfrac{\sigma^2\Delta t+\nu^2(\Delta t)^2}{(\Delta x)^2}+\tfrac{\nu\Delta t}{\Delta x}\Bigr],\quad p_m=1-\tfrac{\sigma^2\Delta t+\nu^2(\Delta t)^2}{(\Delta x)^2},\quad p_d=\tfrac12\Bigl[\tfrac{\sigma^2\Delta t+\nu^2(\Delta t)^2}{(\Delta x)^2}-\tfrac{\nu\Delta t}{\Delta x}\Bigr].$$

Use $\Delta x\approx\sigma\sqrt{3\Delta t}$. **Why trinomial?** Binomial with fixed $\Delta x$ has only one free probability but two constraints (mean and variance).


## 5. Time-dependent volatility (L2.29) — Practice-Final P2

Model: $dS_t = rS_t\,dt + \sigma(t)S_t\,dW_t$ with $\sigma(t)$ known (piecewise constant in P2).

### Where the formula comes from

$d\log S_t = (r-\tfrac12\sigma(t)^2)dt + \sigma(t)\,dW_t$. Integrating, $\log S_T$ is normal because $\int_0^T \sigma(t)\,dW_t$ is a sum of independent normals with variances $\sigma(t)^2 dt$.

$$\boxed{\bar\sigma_T = \sqrt{\frac{1}{T}\int_0^T \sigma(t)^2\,dt}}$$

**Plain English:** $\bar\sigma_T$ is the root-mean-square volatility — the single constant vol that produces the same total variance over $[0,T]$.

### Consequences

$$C_0(K,T)=C^{\mathrm{BS}}(S_0,K,T,r,\bar\sigma_T), \qquad \sigma_{\mathrm{imp}}(K,T)=\bar\sigma_T.$$

**Forward implied vol at time $t$** (for expiry $T>t$):
$$\bar\sigma_{t\to T}=\sqrt{\frac{1}{T-t}\int_t^T \sigma(s)^2\,ds}.$$


In [4]:
def rms_vol(sigma_fn, t0, T):
    ts=np.linspace(t0,T,5001)
    dur=T-t0
    integrand=np.array([sigma_fn(t)**2 for t in ts])
    return np.sqrt(trapz(integrand,ts)/dur)

def sigma_pf2(t):
    return 0.8 if (0.11<=t<=0.12) or (0.36<=t<=0.37) else 0.2

print('Time-0 RMS vol by expiry:')
for T in [0.08,0.16,0.24,0.32,0.40]:
    print(f'  T={T:.2f}: {rms_vol(sigma_pf2,0,T):.4f}')

T_exp=0.24
ts=np.linspace(0,0.235,236)
ivs=[rms_vol(sigma_pf2,t,T_exp) for t in ts]
print(f'\n(c) max forward IV at t={ts[np.argmax(ivs)]:.2f}')
print(f'(d) forward IV at t=0.11: {rms_vol(sigma_pf2,0.11,0.24):.4f}')

Time-0 RMS vol by expiry:
  T=0.08: 0.2000
  T=0.16: 0.2785
  T=0.24: 0.2549
  T=0.32: 0.2426
  T=0.40: 0.2646

(c) max forward IV at t=0.11
(d) forward IV at t=0.11: 0.2935


## 6. Practice-Final P2 — answers

$\sigma(t)=0.8$ on $[0.11,0.12]\cup[0.36,0.37]$, else $0.2$. $S_0=100$, $r=0$, ATM calls.

**(a) Highest time-0 IV:** expiry **0.16** ($\bar\sigma\approx 0.278$). One spike in a short window maximizes the average.

**(b) IV at $T=0.24$:**
$$\bar\sigma^2=\frac{0.8^2\cdot0.01+0.2^2\cdot0.23}{0.24}=0.065,\quad \bar\sigma\approx 0.255.$$

**(c) Time $t\in[0,0.24]$ with highest forward IV:** **$t=0.11$** (just before spike enters remaining life).

**(d) At $t=0.11$, remaining $=0.13$:**
$$\bar\sigma=\sqrt{\frac{0.8^2\cdot0.01+0.2^2\cdot0.12}{0.13}}\approx 0.29.$$


## 7. HW2 P2 — interpolate variance, not vol

Given ATM call prices $C(0.5)=11.25$, $C(1.0)=12.00$, $S_0=100$, $r=0$.

Invert BS to get IVs, then for $T=0.75$ use:
$$\sigma_{0.75}^2\cdot 0.75 = \tfrac12(\sigma_{0.5}^2\cdot 0.5 + \sigma_{1.0}^2\cdot 1.0).$$

Linear interpolation of **vol** (not variance) misprices the 0.75 call → **arbitrage**.


In [5]:
def iv(C,S,K,T,r=0):
    return brentq(lambda s: bs_call(S,K,T,r,s)-C,1e-6,3.0)
iv05=iv(11.25,100,100,0.5); iv10=iv(12.0,100,100,1.0)
var075=0.5*(iv05**2*0.5)+0.5*(iv10**2*1.0)
iv075=np.sqrt(var075/0.75)
print(f'IV(0.5)={iv05:.4f}, IV(1.0)={iv10:.4f}')
print(f'Correct IV(0.75)={iv075:.4f}, price={bs_call(100,100,0.75,0,iv075):.4f}')
print(f'Wrong (linear vol) price={bs_call(100,100,0.75,0,0.5*(iv05+iv10)):.4f}')

IV(0.5)=0.4001, IV(1.0)=0.3019
Correct IV(0.75)=0.3379, price=11.6312
Wrong (linear vol) price=12.0815


## 8. Exercises & cheat sheet

**E1.** $\sigma=0.5$ on $[0,0.1]$, else $0.1$. Find $\bar\sigma_{0.15}$.

**E2.** Why is max forward IV at $t=0.11$ not $t=0.36$ for the 0.24-expiry call?

<details><summary>E1 solution</summary>$\bar\sigma^2=(0.1\cdot0.25+0.05\cdot0.01)/0.15=0.17$, $\bar\sigma\approx0.412$.</details>

<details><summary>E2 solution</summary>The call expires at 0.24, before the second spike at 0.36. That spike is irrelevant.</details>

| Formula | Use |
|---|---|
| $\bar\sigma_T=\sqrt{\frac1T\int_0^T\sigma(t)^2dt}$ | P2 |
| $\sigma_{\mathrm{imp}}=\bar\sigma_T$ when $\sigma=\sigma(t)$ | P2 |
| Interpolate $\sigma^2T$ not $\sigma$ | HW2 P2 |
| $C_n^j=e^{-r\Delta t}E[C_{n+1}]$ | Tree pricing |

*Next: Topic C — Explicit Finite Differences*
